# Anomaly detection with isolation forest in census data
Theodore Charm
- This notebook illustrates how I use the Isolation Forest algorithm to detect anomalies in census data. It explores the 2024 American Community Survey (ACS) 5-year data to identify the census tracts that have exceptionally high or low economic well-being--- which I consider as anomalies.

# Introduction
Using R's tidycensus package, I explore the 2024 ACS 5-year estimates of all census tracts across the U.S. I extract the following variables from the ACS data at the census tract level:
- mhi: This measures the median household income (B19019_001)
- pci: This measures the per capita income (B19301_001)
- home_value: This measures the median home value (B25077_001):
- college_perc: This computes the percentage of residents aged 25 and over with Bachelor's degree or above (B15003)

This notebook applies the Isolation Forest algorithm to detect census tracts that have exceptionally high or low economic well-being, which I consider as anomalies. Depending on the research goals, these census tracts can be kept or removed for downstream analysis.

Isolation Forest is an unsupervised machine learning algorithm designed specifically for anomaly and outlier detection within a dataset. It goes through the following pipeline:
1. Select a random subset of data from the dataset
2. Pick a random feature from the data. Pick a random split value between the minimum and maximum values of the feature
3. The data subset is split into two branches conditional on each data point's feature value. This partitioning is repeated recursively until every data point is isolated in its leaf node
4. Measure the "path length" (number of splits required to separate a data point from the root node)
5. Repeat the process to create an ensemble of randomized trees
6. Average the "path length" of a data point across the entire forest to calculate a standardized anomaly score ranging from -1 to 1
7. Data points with anomaly score close to -1 isolate quickly, hence flagged as anomalies

# Setup
## Packages

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# Data preprocessing
## Import dataset

In [2]:
df = pd.read_csv("census_data.csv")
df.shape

(84401, 6)

- In 2024, there are 84,401 census tracts across the U.S.

In [3]:
# Display the first 5 rows
df.head()

,GEOID,tract,mhi,pci,home_value,college_perc
0,1001020100,Census Tract 201; Autauga County; Alabama,63237.0,32302.0,188800.0,25.01883949
1,1001020200,Census Tract 202; Autauga County; Alabama,61229.0,31064.0,154900.0,19.63898917
2,1001020300,Census Tract 203; Autauga County; Alabama,63939.0,27091.0,147700.0,12.18450827
3,1001020400,Census Tract 204; Autauga County; Alabama,69348.0,45895.0,181700.0,27.45858301
4,1001020501,Census Tract 205.01; Autauga County; Alabama,88965.0,47982.0,199000.0,47.64870757


In [4]:
df.dtypes

,0
GEOID,int64
tract,object
mhi,float64
pci,float64
home_value,float64
college_perc,object


In [5]:
# Convert college_perc to numerical values
df['college_perc'] = pd.to_numeric(df['college_perc'], errors='coerce')

In [6]:
# Count number of missing values for each column
df.isnull().sum()

,0
GEOID,0
tract,0
mhi,1488
pci,857
home_value,3145
college_perc,808


- There are relatively few missingness across the features. This has a negligible impact on the Isolation Forest's performance.

# Analysis

In [7]:
# Initialize the Isolation Forest model
# 'contamination' sets the expected proportion of outliers in the dataset, which I set to be 1%
model = IsolationForest(contamination=0.01, random_state=10)

In [8]:
# Select mhi, pci, home_value, college_perc as features
# Fit the model and predict anomalies
# Predict returns 1 for normal data (inliers) and -1 for anomalies (outliers)
df['Anomaly_Class'] = model.fit_predict(df[['mhi', 'pci', 'home_value', 'college_perc']])

In [9]:
# Get raw anomaly scores
# Negative scores indicate outliers; higher positive scores indicate normal data
df['Anomaly_Score'] = model.decision_function(df[['mhi', 'pci', 'home_value', 'college_perc']])

# Findings

In [10]:
# Filter and review the detected anomalies
# Take a random sample of 20 census tracts
anomalies = df[df['Anomaly_Class'] == -1]
anomalies.sample(n=20, random_state = 20)

,GEOID,tract,mhi,pci,home_value,college_perc,Anomaly_Class,Anomaly_Score
15572,9190035300,Census Tract 353; Western Connecticut Planning...,250001.0,122991.0,1579400.0,83.794758,-1,-0.014900
75366,48339690605,Census Tract 6906.05; Montgomery County; Texas,250001.0,126754.0,665400.0,82.822086,-1,-0.014480
10043,6067988300,Census Tract 9883; Sacramento County; California,232700.0,5311.0,NaN,2.139937,-1,-0.016197
4829,6013355113,Census Tract 3551.13; Contra Costa County; Cal...,250001.0,100895.0,1925100.0,68.873128,-1,-0.019101
52500,36061016700,Census Tract 167; New York County; New York,250001.0,154052.0,2000001.0,90.358974,-1,-0.042779
4837,6013355124,Census Tract 3551.24; Contra Costa County; Cal...,250001.0,103100.0,1786000.0,86.029936,-1,-0.024939
73737,48201411100,Census Tract 4111; Harris County; Texas,177796.0,155963.0,1745500.0,79.185693,-1,-0.009992
4518,6001443105,Census Tract 4431.05; Alameda County; California,248185.0,98705.0,2000001.0,76.695157,-1,-0.013656
25847,17031800200,Census Tract 8002; Cook County; Illinois,248893.0,142409.0,1186500.0,89.424008,-1,-0.016037
12066,6085502601,Census Tract 5026.01; Santa Clara County; Cali...,193125.0,130253.0,2000001.0,63.564356,-1,-0.003950


- This table displays a random sample of 20 census tracts predicted as anomalies. We can see that most of these anomalies have exceptionally high economic well-being, including median household income and median home value significantly higher than the nationwide average.
- Census tract 9883 of Sacramento County, CA, is an interesting case. It has very high MHI but very low PCI and college-educated population.

# Conclusions
- Isolation Forest algorithm can detect anomalies in large-scale census data by isolating anomalies while randomly splitting features
- Isolation Forest is an efficient machine learning algorithm for detecting socio-economic extremes for census tracts across the U.S.